# Flash Sales: Within-City Difference-in-Differences

## The logic

Maverick cities (Dortmund, Dresden, Essen) have 6+ concurrent campaigns running continuously (Smart Growth, SVP, Win Back, etc.). These are active on **both** Flash Sales days and non-Flash-Sales days.

If we compare dinner-hour orders on **Flash Sales days** to dinner-hour orders on **adjacent non-Flash-Sales days** within the same city, the background campaigns cancel out (they're present in both periods). The difference is the **marginal Flash Sales effect**.

This is a simple Difference-in-Differences where the city is its own control:
- Treatment: Flash Sales dinner hours
- Control: Same dinner hours on adjacent days (same background campaigns, no Flash Sales)

### Advantages
- No control cities needed (eliminates contamination and selection problems)
- Background campaigns cancel out by design
- Simple and transparent

### Limitations
- Very few data points (2 Flash Sales days vs 3-5 comparison days)
- Low statistical power
- Day-of-week effects may confound (if Flash Sales run Fri-Sat but comparison is Mon-Thu)
- Assumes background campaign effect is constant across days

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

from google.colab import auth
from google.cloud import bigquery
auth.authenticate_user()

from google.colab import drive
drive.mount('/content/drive')

import sys, os
CONFIG_DIR = '/content/drive/MyDrive/aa_test'
sys.path.insert(0, CONFIG_DIR)
os.chdir(CONFIG_DIR)

import aa_config as cfg
import flash_sales_utils as fsu

client = bigquery.Client(project=cfg.PROJECT_ID)

# === CONFIGURE FLASH SALES TO ANALYZE ===
COUNTRY = 'DE'
HOUR_START = 16
HOUR_END = 22

# Each entry: (name, cities, flash_dates, comparison_dates)
# Comparison dates = same dinner hours, same city, no Flash Sales,
# but all other Maverick campaigns still running.
# Choose dates close to Flash Sales to minimize temporal drift.
FLASH_SALES_EVENTS = [
    {
        'name': 'Essen Dinner Flash (27-28 Feb)',
        'cities': ['Essen'],
        'flash_dates': ['2026-02-27', '2026-02-28'],
        # Adjacent non-flash dinner days (Mon-Wed before)
        'comparison_dates': ['2026-02-23', '2026-02-24', '2026-02-25', '2026-02-26'],
        # NOTE: Feb 24 had a lunch flash sale, but dinner hours were clean
        # Feb 25-26 had dinner flash sales in Essen/Dortmund — EXCLUDE these
        # Revised: use days BEFORE the first dinner flash of the week
    },
    {
        'name': 'Weekday Flash Dortmund/Essen (4 Mar)',
        'cities': ['Dortmund', 'Essen'],
        'flash_dates': ['2026-03-04'],
        'comparison_dates': ['2026-03-01', '2026-03-02', '2026-03-03'],
    },
    {
        'name': 'Weekend Flash Dortmund/Essen (6-8 Mar)',
        'cities': ['Dortmund', 'Essen'],
        'flash_dates': ['2026-03-06', '2026-03-07', '2026-03-08'],
        'comparison_dates': ['2026-03-01', '2026-03-02', '2026-03-03'],
    },
]

print(f'Flash Sales events to analyze: {len(FLASH_SALES_EVENTS)}')
for e in FLASH_SALES_EVENTS:
    print(f"  {e['name']}: {e['flash_dates']} vs {e['comparison_dates']}")

## 2. Campaign Overlap Check

Before running DiD, verify which background campaigns were active on both Flash Sales and comparison days. They should be the same — if a new campaign started between comparison and Flash Sales dates, the DiD is confounded.

In [ ]:
for event in FLASH_SALES_EVENTS:
    print(f'\n=== {event["name"]} ===')
    for city in event['cities']:
        print(f'\n  {city}:')
        flash_campaigns = set()
        for d in event['flash_dates']:
            for c in fsu.get_concurrent_campaigns(city, d):
                flash_campaigns.add(c['campaign'])

        comp_campaigns = set()
        for d in event['comparison_dates']:
            for c in fsu.get_concurrent_campaigns(city, d):
                comp_campaigns.add(c['campaign'])

        both = flash_campaigns & comp_campaigns
        flash_only = flash_campaigns - comp_campaigns
        comp_only = comp_campaigns - flash_campaigns

        print(f'    Active in both periods: {both}')
        if flash_only:
            print(f'    WARNING - Only on flash days: {flash_only}')
        if comp_only:
            print(f'    WARNING - Only on comparison days: {comp_only}')
        if not flash_only and not comp_only:
            print(f'    CLEAN: Same background campaigns in both periods')

## 3. Pull Hourly Data & Run DiD

In [ ]:
all_results = []

for event in FLASH_SALES_EVENTS:
    print(f'\n{"="*50}')
    print(f'{event["name"]}')
    print(f'{"="*50}')

    all_dates = event['flash_dates'] + event['comparison_dates']
    earliest = min(all_dates)
    latest = max(all_dates)

    for city in event['cities']:
        print(f'\n--- {city} ---')

        # Pull hourly data for this city
        df = fsu.get_hourly_orders_single_city(
            client, COUNTRY, city,
            earliest, latest, HOUR_START, HOUR_END)

        if df.empty:
            print(f'  No data for {city}')
            continue

        print(f'  {len(df)} hourly rows')

        # Run DiD
        result = fsu.run_did_analysis(
            df, event['flash_dates'], event['comparison_dates'])

        if result:
            result['city'] = city
            result['event'] = event['name']
            all_results.append(result)

            print(f'  Flash days mean:      {result["flash_mean_orders"]:.0f} orders')
            print(f'  Comparison days mean: {result["pre_mean_orders"]:.0f} orders')
            print(f'  Difference:           {result["diff_orders"]:+.0f} orders '
                  f'({result["pct_diff"]:+.1%})')
            print(f'  p-value:              {result["p_value"]:.4f}'
                  if not np.isnan(result["p_value"]) else '  p-value: N/A (too few points)')
            if not np.isnan(result['ci_lower']):
                print(f'  95% CI:               [{result["ci_lower"]:+.0f}, '
                      f'{result["ci_upper"]:+.0f}] orders')

## 4. Summary

In [ ]:
if all_results:
    df_results = pd.DataFrame(all_results)

    print('=' * 70)
    print('FLASH SALES DiD RESULTS: MARGINAL EFFECT')
    print('=' * 70)

    for _, r in df_results.iterrows():
        sig = '***' if r['p_value'] < 0.01 else '**' if r['p_value'] < 0.05 else '*' if r['p_value'] < 0.10 else ''
        print(f'{r["event"]} | {r["city"]}:')
        print(f'  {r["diff_orders"]:+.0f} orders ({r["pct_diff"]:+.1%}) {sig}')

    # Overall pooled estimate
    total_flash = df_results['flash_mean_orders'].sum()
    total_pre = df_results['pre_mean_orders'].sum()
    total_diff = total_flash - total_pre
    total_pct = total_diff / total_pre if total_pre > 0 else np.nan

    print(f'\nPooled across all events:')
    print(f'  Flash days total:      {total_flash:.0f}')
    print(f'  Comparison days total: {total_pre:.0f}')
    print(f'  Marginal difference:   {total_diff:+.0f} ({total_pct:+.1%})')
    print()
    print('NOTE: This is the MARGINAL Flash Sales effect on top of')
    print('always-running background campaigns. It is NOT the total')
    print('Flash Sales effect (which would require a clean control).')
    print('=' * 70)
else:
    print('No DiD results computed.')

## 5. Visualization

In [ ]:
if all_results:
    df_results = pd.DataFrame(all_results)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Bar chart: flash vs comparison per event/city
    ax = axes[0]
    labels = [f'{r["event"]}\n{r["city"]}' for _, r in df_results.iterrows()]
    x = range(len(labels))
    w = 0.35
    ax.bar([i - w/2 for i in x], df_results['pre_mean_orders'],
           w, label='Comparison days', color='steelblue', alpha=0.7)
    ax.bar([i + w/2 for i in x], df_results['flash_mean_orders'],
           w, label='Flash Sales days', color='tomato', alpha=0.7)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=8, rotation=15)
    ax.set_ylabel('Mean Daily Orders (dinner hours)')
    ax.set_title('Flash Sales vs. Comparison Days')
    ax.legend()

    # Difference chart
    ax = axes[1]
    colors = ['green' if d > 0 else 'red'
              for d in df_results['diff_orders']]
    ax.barh(range(len(labels)), df_results['diff_orders'],
            color=colors, alpha=0.7, edgecolor='black')
    if 'ci_lower' in df_results.columns:
        for i, (_, r) in enumerate(df_results.iterrows()):
            if not np.isnan(r['ci_lower']):
                ax.plot([r['ci_lower'], r['ci_upper']], [i, i],
                        'k-', linewidth=2)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, fontsize=8)
    ax.axvline(0, color='black', linewidth=0.5)
    ax.set_xlabel('Difference in Orders')
    ax.set_title('Marginal Flash Sales Effect (DiD)')

    plt.tight_layout()
    plt.show()

## 6. Interpretation

### What this tells us
The DiD measures the **marginal spike** in dinner-hour orders on Flash Sales days compared to adjacent days when the same background campaigns were running. If Flash Sales add value, we should see a positive, consistent spike.

### What this does NOT tell us
- The **total** Flash Sales effect (background campaigns may interact with Flash Sales)
- Whether Flash Sales are cost-effective (we don't have cost data here)
- Whether the effect persists after Flash Sales end

### Statistical caveat
With only 2-3 Flash Sales days and 3-5 comparison days, the statistical power is very low. Treat the results as **directional evidence**, not definitive proof. A non-significant result does not mean "no effect."

### Comparison day selection matters
The comparison dates must have the same background campaigns active. If a new campaign launched between the comparison and Flash Sales dates, the DiD is confounded. Check the campaign overlap output in Section 2.